# Getting Started with MPSFast.jl

This notebook takes you from zero to a trained **Matrix Product State (MPS) Born machine** in five steps:

1. Install / activate the package
2. Generate or load some paths
3. Encode the paths
4. Train the MPS
5. Sample new paths and check the fit

No prior knowledge of quantum mechanics is required — think of an MPS as a very expressive, tractable **joint probability model** over discrete sequences.

## 1. Setup

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # MPSFast.jl project root
Pkg.resolve()      # writes/updates Manifest.toml (handles stdlib deps correctly)
Pkg.instantiate()  # downloads/installs any missing registered packages

  Activating project at `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl`
     Project No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Project.toml`
    Manifest No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Manifest.toml`


In [2]:
using MPSFast
using MPSFast.Encoders
using Random, LinearAlgebra, Statistics, Printf

println("Threads: ", Threads.nthreads())

Threads: 4


## 2. Generate synthetic paths

We use a simple **Geometric Brownian Motion** (GBM) for illustration.
Replace with real data or a Heston simulation from `experiments/paper_reproduction.ipynb`.

In [3]:
rng = MersenneTwister(42)

N = 5_000    # number of paths
M = 20       # timesteps per path
dt = 1/252
μ  = 0.05
σ  = 0.20
S0 = 100.0

# GBM: S[t+1] = S[t] * exp((μ - σ²/2)dt + σ√dt Z)
Z = randn(rng, N, M)
log_returns = (μ - 0.5 * σ^2) * dt .+ σ * sqrt(dt) .* Z
log_prices  = cumsum(log_returns, dims = 2)
paths       = S0 .* exp.(log_prices)   # (N, M)

println("paths: ", size(paths), "  min=", round(minimum(paths), digits=2),
        "  max=", round(maximum(paths), digits=2))

paths: (5000, 20)  min=81.42  max=124.03


## 3. Encode paths

We discretise each continuous price level into one of `d = 2^m` buckets.
`BasisEncoder` uses one MPS site per timestep with `d`-dimensional physical indices.

In [4]:
m   = 4                        # d = 2^4 = 16 buckets
enc = BasisEncoder(m)
fit_grid!(enc, paths)          # calibrate Smin/Smax to training data

xi = encode_paths(enc, paths)  # (N, M) integer matrix, values in 1:16

encoder_summary(enc, M, 16)    # print a quick summary
println("\nxi shape: ", size(xi), "  unique values: ", sort(unique(xi)))

┌ Info: Encoder
│   encoder = BasisEncoder
│   M = 20
│   chain_length = 20
│   site_dim = 16
│   D_max = 16
│   params = 74240
└ @ MPSFast.Encoders /Users/bi006881/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/src/Encoders.jl:414



xi shape: (5000, 20)  unique values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]


## 4. Initialise and train the MPS

The MPS is trained by minimising the **Negative Log-Likelihood** (NLL) via a
DMRG-style sweep with Adam updates at each bond. Each epoch = one forward sweep + one backward sweep.

In [5]:
D_max    = 20      # maximum bond dimension
n_epochs = 30      # 10 was too few; 30 shows clear convergence
η        = 3e-3    # Adam learning rate (5e-4 was too small to see progress in 10 epochs)
ε_cut    = 1e-5    # SVD truncation floor

mps = init_mps(chain_length(enc, M), site_dim(enc), D_max; rng = MersenneTwister(1))

nll_hist = train_mps!(
    mps, xi, n_epochs, η, D_max, ε_cut;
    verbose      = true,
    nll_samples  = 500,
)

train_mps!: Ml=20, Nd=5000, d=16, D_max=20, epochs=30, one-hot
— Epoch 1/30 —
  ↳ norm envs ready → forward sweep (19 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …
  ↳ backward done → canonicalize + NLL estimate …
Epoch 1/30 | NLL ≈ 26.7321 | η=0.003 | bonds=[16,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,16] | 0.566 s
— Epoch 2/30 —
  ↳ norm envs ready → forward sweep (19 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …
  ↳ backward done → canonicalize + NLL estimate …
Epoch 2/30 | NLL ≈ 21.1045 | η=0.003 | bonds=[16,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,16] | 0.264 s
— Epoch 3/30 —
  ↳ norm envs ready → forward sweep (19 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …
  ↳ backward done → canonicalize + NLL estimate …
Epoch 3/30 | NLL ≈ 19.8403 | η=0.003 | bonds=[16,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,16] | 0.262 s
— Epoch 4/30 —
  ↳ norm envs ready → forward sweep (19 

30-element Vector{Float64}:
 26.732076373226572
 21.104504168393973
 19.840303256330188
 19.659820723058612
 19.3052165817131
 19.14399775856337
 19.32895647577965
 19.062163375424195
 18.710685501568424
 18.844037447907464
  ⋮
 17.93883502027203
 17.762591086354192
 17.49239743382371
 17.4412949327818
 17.330752859405052
 17.496600793607954
 17.429754367163522
 17.579603645706236
 17.687458115896536

In [6]:
# NLL learning curve
println("NLL history: ", round.(nll_hist, digits = 4))
println("Final NLL  : ", round(nll_hist[end], digits = 4))

NLL history: [26.7321, 21.1045, 19.8403, 19.6598, 19.3052, 19.144, 19.329, 19.0622, 18.7107, 18.844, 18.8354, 18.4039, 18.5687, 18.2395, 18.1322, 18.0296, 18.0374, 17.8853, 17.8215, 17.6518, 17.8177, 17.9388, 17.7626, 17.4924, 17.4413, 17.3308, 17.4966, 17.4298, 17.5796, 17.6875]
Final NLL  : 17.6875


## 5. Sample and validate

Draw new paths by sequential conditional sampling and compare marginal statistics
to the training data.

In [7]:
n_samp = 2_000
sampled_paths, sampled_xi = sample_paths(enc, mps, n_samp; seed = 99)

println("Sampled paths shape: ", size(sampled_paths))

# Compare per-timestep mean and std
println("\nTimestep  Train mean  Samp mean  Train std  Samp std")
for t in [1, 5, 10, 15, 20]
    @printf("  t=%-3d    %8.3f   %8.3f   %8.3f  %8.3f\n", t, mean(paths[:,t]), mean(sampled_paths[:,t]), std(paths[:,t]), std(sampled_paths[:,t]))
end

Sampled paths shape: (2000, 20)

Timestep  Train mean  Samp mean  Train std  Samp std
  t=1       100.004     98.604      1.257     2.538
  t=5       100.053     98.742      2.816     3.315
  t=10      100.112     98.722      3.979     4.195
  t=15      100.216     98.854      4.870     5.105
  t=20      100.255     98.845      5.611     5.740


In [8]:
# Bond dimensions after training
bonds = [size(mps[j], 3) for j in 1:length(mps)-1]
println("Bond dimensions: ", bonds)

Bond dimensions: [16, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 16]


In [9]:
# Bipartite von-Neumann entropy profile
Svals, entropies = bipartite_entropies(mps)
println("Bipartite entropies (nats): ", round.(entropies, digits = 3))

Bipartite entropies (nats): [0.587, 1.098, 1.396, 1.595, 1.359, 1.498, 1.544, 1.653, 1.822, 1.856, 1.886, 1.888, 1.911, 1.857, 1.989, 1.939, 2.016, 2.06, 1.878]


## Next steps

- **`notebooks/dmrg_tutorial.ipynb`** — step-by-step derivation of the DMRG algorithm.
- **`notebooks/experiments/paper_reproduction.ipynb`** — full Heston model reproduction and options pricing.
- **`notebooks/experiments/encodings_comparison.ipynb`** — compare BasisEncoder vs BinaryEncoder vs TrigEncoder.
- **`notebooks/experiments/classification.ipynb`** — supervised Born-machine classification.